# Frequency analysis
End-to-end notebook for CBS measurement frequencies, four-hour threshold sensitivity, exports, and frequency visualisation. Reusable functions are imported from the project modules.

In [ ]:
import geopandas as gpd

from Preparation import process_realtime_with_cbs
from Optimization import frequency_threshold_table, pipeline_plot_frequency


## 1. Prepare hourly CBS frequency counts

In [ ]:
# Update paths to the selected-vehicle point dataset used for the analysis.
gdf_cbs = gpd.read_file("cbs_full.shp")
points_realtime = gpd.read_file("data/snapped_ams_1503_no_duplicates.gpkg")

grouped_by_points_GVB, cbs_interval_counts_GVB = process_realtime_with_cbs(
    gdf_cbs,
    points_realtime,
    buffer_size=50,
)

print(cbs_interval_counts_GVB.columns.tolist())


## 2. Measurement-frequency sensitivity
For each optimization strategy, calculate the share of CBS cells sensed at least once in the selected hour that reach the minimum threshold of 12 measurements per hour. Hours: 07-08, 13-14, 19-20, and 01-02.

In [ ]:
# Add each optimization's cbs_interval_counts object here after preparing it.
strategy_interval_counts = {
    "Current selection": cbs_interval_counts_GVB,
    # "Fairness": cbs_interval_counts_fairness,
    # "Spatial": cbs_interval_counts_spatial,
    # "Temporal": cbs_interval_counts_temporal,
    # "Combined": cbs_interval_counts_combined,
}

frequency_sensitivity_long, frequency_sensitivity_table = frequency_threshold_table(
    strategy_interval_counts,
    hours=("7-8", "13-14", "19-20", "1-2"),
    threshold_per_hour=12,
)

display(frequency_sensitivity_table.round(1))
display(frequency_sensitivity_long.round(1))

frequency_sensitivity_table.round(1).to_csv(
    "data/frequency_sensitivity_threshold12.csv", index=False
)
frequency_sensitivity_long.round(1).to_csv(
    "data/frequency_sensitivity_threshold12_detailed.csv", index=False
)


## 3. Frequency visualisation

In [ ]:
ams_gdf = gpd.read_file("data/Gemeente2.geojson")

column_to_plot = "7-8"
threshold = 12

interval_counts_CBS_data, fig_frequency, fig_frequency_th, fig_freq_person = (
    pipeline_plot_frequency(
        cbs_interval_counts_GVB,
        gdf_cbs,
        ams_gdf,
        column_to_plot,
        threshold,
    )
)

display(fig_frequency)
display(fig_frequency_th)
display(fig_freq_person)
